In [16]:
from pathlib import Path
import numpy as np
from tqdm import tqdm
import shutil
import math
from ase.structure import graphene_nanoribbon


In [17]:
def leggi_file_xyz(nome_file:Path):
    """
    Legge un file XYZ e restituisce le coordinate degli atomi.
    """
    coordinate = []
    with open(str(nome_file), 'r') as file:
        num_atom = int(next(file))
        # Ignora la prima riga (commento)
        next(file)
        # Leggi il numero di atomi
            
        for _ in range(num_atom):
            atom, x, y, z = next(file).split()
            coordinate.append((atom, float(x), float(y), float(z)))
    return coordinate

def trasla_atom(file_name: Path, file_out: Path, cell_x:float=34.43317005, cell_y:float=34.08, delta_x:float=0.4, delta_y:float=0.4):
    """
    Trasla gli atomi con coordinata x minore di 0.1 di una quantità specificata.
    """
    coordinate = leggi_file_xyz(file_name)
    coordinate_modificate = []
    for atom, x, y, z in coordinate:
        if cell_x - x < delta_x:
            x -= cell_x
        if cell_y - y < delta_y:
            y -= cell_y
        coordinate_modificate.append((atom, x, y, z))
        
    with open(str(file_out), 'w') as file:
        file.write(f"{len(coordinate_modificate)}\n")
        file.write("Atoms\n")
        for atom, x, y, z in coordinate_modificate:
            file.write(f"{atom} {x:.6f} {y:.6f} {z:.6f}\n")

In [18]:
def check_num_atoms(nome_file:Path, box_y:float=34.08, delta:float=1.88):
    num = int(box_y/(1.5*1.42))*2
    coordinate = leggi_file_xyz(nome_file)
    coordinate_x = []
    for atom, x, y, z in coordinate:
        coordinate_x.append(x)
    
    x_left = [x for x in coordinate_x if min(coordinate_x) <= x <= min(coordinate_x)+delta]
    x_right = [x for x in coordinate_x if max(coordinate_x)-delta<= x <= max(coordinate_x)]
    
    if len(x_left) == len(x_right) ==  num:
        return True
    else:
        return False

In [19]:
xyz_files = Path("/home/cnrismn/git_workspace/dftb/data/transport/xyz_files")
fixed_path=Path("/home/cnrismn/git_workspace/dftb/data/transport/fixed")
fixed_path.mkdir(exist_ok=True, parents=True)
broken_path=Path("/home/cnrismn/git_workspace/dftb/data/transport/broken")
broken_path.mkdir(exist_ok=True, parents=True)
samples = [f for f in xyz_files.iterdir() if f.suffix.lower() == ".xyz"]
for sample in tqdm(samples):
    trasla_atom(sample,fixed_path.joinpath(f"{sample.stem}_fixed.xyz"))
    
samples = [f for f in fixed_path.iterdir() if f.suffix.lower() == ".xyz"]
for sample in tqdm(samples):
    if not check_num_atoms(sample):
        shutil.copy(sample,broken_path.joinpath(sample.name))

100%|██████████| 43/43 [00:00<00:00, 318.57it/s]


In [21]:
from ase.structure import graphene_nanoribbon

def get_electrode(out_path:Path, x_len:int=3, y_len:int=8, type:str="armchair"):
    # Genera un foglio di grafene ideale
    graphene_sheet = graphene_nanoribbon(x_len,y_len, type, sheet=True)
    # Salva la struttura in un file XYZ
    graphene_sheet.write(out_path)

    def rotate_structure(input_file, output_file, rotation_matrix):
        with open(input_file, 'r') as f_in:
            lines = f_in.readlines()

        # Get the number of atoms and the cell parameters from the input file
        num_atoms = int(lines[0])

        # Extract atomic positions
        atoms = []
        for line in lines[2:2 + num_atoms]:
            atoms.append(list(map(float, line.split()[1:])))

        # Convert atomic positions to numpy array for matrix multiplication
        atoms = np.array(atoms)

        # Apply rotation to atomic positions
        rotated_atoms = np.dot(atoms, rotation_matrix)

        # Write the rotated structure to the output file
        with open(output_file, 'w') as f_out:
            f_out.write(f"{num_atoms}\n")
            f_out.write(lines[1])  # Copy comment line from the input file
            for i in range(num_atoms):
                atom_line = " ".join(map(str, [lines[i + 2].split()[0]] + list(rotated_atoms[i]))) + "\n"
                f_out.write(atom_line)

    # Usage example
    rotation_matrix = np.array([[1, 0, 0],
                                [0, 0, -1],
                                [0, 1, 0]])  # Example rotation matrix (90 degree rotation around y-axis)
    rotate_structure(out_path, out_path, rotation_matrix)

In [23]:
def generate_electrode(xyz_fixed_files:Path, out_path: Path, electrode_path:Path=Path("/home/cnrismn/git_workspace/dftb/data/transport/elettrodi/electrode.xyz")):
    
    coordinate_elettrodo = leggi_file_xyz(electrode_path)
    coordinate_x = []
    for atom, x, y, z in coordinate_elettrodo:
        coordinate_x.append(x)

    offset = max(coordinate_x)+1.42*math.cos(math.pi/6)
    
    coordinate_modificate = []
    for atom, x, y, z in coordinate_elettrodo:
        coordinate_modificate.append((atom, x, y, z))
        
    coordinate = leggi_file_xyz(xyz_fixed_files)
    for atom, x, y, z in coordinate:
        x+=offset
        coordinate_modificate.append((atom, x, y, z))
        
    for atom, x, y, z in coordinate_elettrodo:
        x+=offset+34.43317005
        coordinate_modificate.append((atom, x, y, z))
        
    with open(str(out_path), 'w') as file:
        file.write(f"{len(coordinate_modificate)}\n")
        file.write("Atoms\n")
        for atom, x, y, z in coordinate_modificate:
            file.write(f"{atom} {x:.6f} {y:.6f} {z:.6f}\n")

In [24]:
generate_electrode(xyz_fixed_files=Path("/home/cnrismn/git_workspace/dftb/data/transport/fixed/graphene_2465_fixed.xyz"), out_path=Path("/home/cnrismn/git_workspace/dftb/data/transport/elettrodi/graphene_2465_l.xyz"),electrode_path=Path("/home/cnrismn/git_workspace/dftb/data/transport/elettrodi/electrode.xyz"))